# Phase 1: TD(0) Value Learning — Random Policy (Partner Replication)

**Goal**: Learn $V_i(s)$ for each agent under random policy, validate against MC ground truth.
Batch TD(0) with partner's 2-mode transition system.

**Setup**: Run `sbatch run_mc.sh` first to generate MC ground truth, then Run All here.


In [1]:

# =============================================================================
# CONFIG (edit these, then Run All)
# =============================================================================

R_PICKER = -1.0
LAMBDA = 0.8               # 0 = TD(0)
MLP_LAYERS = [16, 16, 16, 16]

PHASE2_EVAL = False # ep greedy or random policy
POLICY_EVAL_SECS = 100
POLICY_EVAL_EPISODES = 1
EPSILON_START = 1
EPSILON_END = 0.1
EPSILON_DECAY_FRAC = 0.4


TD_MODE = "tdlambda"           # "td0" = original code, "tdlambda" = sliding window TD(λ)
WINDOW_SEC = 10            # λ-return lookback in seconds (only used if TD_MODE="tdlambda")

SEMI_MDP = True

LR = 0.01
LR_MIN = 1e-6
LR_DECAY_FRAC = 0.8

TRAIN_SECONDS = 1_000_000     # sections (rounds). Each = all 4 agents × 2 modes.
EVAL_FREQ = 10
CHECKPOINT_FREQ = 20_000
PRINT_FREQ = 10_000

ADD_GRAD_CLIP = False
CLIP_GRAD_NORM = 10000
CLAMP_TARGETS = False

MC_DEPTH = 1000
MC_TRAJ = 5
MC_ROLLOUTS = 200
NUM_MC_STATES = 10

MC_CACHE_DIR = "mc_cache_inline"

SEED = 42069
OUTPUT_PATH = "outputs"
TAG = f"td_lam{LAMBDA}_lr{LR}_rp{R_PICKER}_semi{SEMI_MDP}"


In [2]:
# =============================================================================
# IMPORTS + DERIVED CONSTANTS
# =============================================================================

import os, sys, json, time, glob
import numpy as np
import pandas as pd
import torch

from config import (
    H, W, NUM_AGENTS, GAMMA, GAMMA_STEP, GAMMA_SEMI, K_NEAREST, INPUT_DIM,
    P_SPAWN, P_DESPAWN, L,
)
from helpers import (
    set_all_seeds, Reward, Orchard,
    encode_state, ValueNetwork, lr_factor,
    transition, evaluate_on_mc,
    get_memory_mb, get_weight_stats,
    evaluate_policies, greedy_action, random_action,
)
import random
torch.set_default_dtype(torch.float64)

start_time = time.time()
R_OTHER = (1.0 - R_PICKER) / (NUM_AGENTS - 1)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Device: {device}")
print(f"γ_round = {GAMMA}, γ_step = {GAMMA_STEP:.6f}")
print(f"input_dim = {INPUT_DIM}")
print(f"P_spawn = {P_SPAWN:.6f}, P_despawn = {P_DESPAWN:.6f}")
print(f"R_picker = {R_PICKER}, R_other = {R_OTHER:.4f}")
print(f"Each second = 1 section = {NUM_AGENTS} agent-steps × 2 modes = {2*NUM_AGENTS} transitions/agent")

TAG += f"gamma{GAMMA}"


Device: cpu
γ_round = 0.99, γ_step = 0.998744
input_dim = 55
P_spawn = 0.012346, P_despawn = 0.031250
R_picker = -1.0, R_other = 0.6667
Each second = 1 section = 4 agent-steps × 2 modes = 8 transitions/agent


In [3]:
# =============================================================================
# GENERATE MC GROUND TRUTH (inline, cached)
# =============================================================================
import copy, random, glob
from pathlib import Path
from helpers import mc_ground_truth

if not PHASE2_EVAL:
    # Cache path encodes all MC-relevant parameters
    mc_tag = (f"rp{R_PICKER}_g{GAMMA}_N{NUM_AGENTS}_d{MC_DEPTH}"
            f"_t{MC_TRAJ}_r{MC_ROLLOUTS}_seed{SEED}")
    mc_cache_path = Path(MC_CACHE_DIR) / mc_tag
    mc_cache_path.mkdir(parents=True, exist_ok=True)

    # Load all existing cached states
    mc_states = []
    for fpath in sorted(mc_cache_path.glob("state_*.npz"), key=lambda p: int(p.stem.split("_")[1])):
        with np.load(fpath, allow_pickle=True) as data:
            st_mc = data["state_dict"].item()
        mc_states.append(st_mc)
        print(f"Loaded {fpath.name}")

    num_cached = len(mc_states)
    print(f"Found {num_cached} cached states in {mc_cache_path}")

    # Generate more if we have fewer than NUM_MC_STATES
    if num_cached < NUM_MC_STATES:
        # Find max existing index to avoid collisions
        existing_indices = set()
        for fpath in mc_cache_path.glob("state_*.npz"):
            existing_indices.add(int(fpath.stem.split("_")[1]))
        next_idx = max(existing_indices, default=-1) + 1

        for count in range(NUM_MC_STATES - num_cached):
            idx = next_idx + count
            set_all_seeds(SEED + idx)
            reward_mod_mc = Reward(R_PICKER, NUM_AGENTS)
            orch_mc = Orchard(W, W, NUM_AGENTS, reward_mod_mc, p_apple=0.2, d_apple=P_DESPAWN)
            orch_mc.p_apple = P_SPAWN
            orch_mc.set_positions()
            st_mc = dict(orch_mc.get_state())
            st_mc["actor_id"] = random.randint(0, NUM_AGENTS - 1)
            st_mc["mode"] = 0

            mc_standard, dist_standard = mc_ground_truth(
                SEED + idx, MC_DEPTH, orch_mc, st_mc, GAMMA_STEP,
                MC_TRAJ, MC_ROLLOUTS, semi_mdp=False,
            )
            mc_semi, dist_semi = mc_ground_truth(
                SEED + idx, MC_DEPTH, orch_mc, st_mc, GAMMA_SEMI,
                MC_TRAJ, MC_ROLLOUTS, semi_mdp=True,
            )

            st_mc["dist_standard"] = dist_standard
            st_mc["dist_semi"] = dist_semi
            st_mc["mc_standard"] = mc_standard
            st_mc["mc_semi"] = mc_semi

            print(f"Generated state_{idx}: apples={st_mc['apples'].sum()}, actor={st_mc['actor_id']}")
            for i in range(NUM_AGENTS):
                print(f"  Agent {i}: standard={mc_standard[i]:.6f}, semi={mc_semi[i]:.6f}")

            fpath = mc_cache_path / f"state_{idx}.npz"
            np.savez_compressed(fpath, state_dict=np.array(st_mc, dtype=object))
            mc_states.append(st_mc)

    # Select active MC values based on SEMI_MDP flag
    for st in mc_states:
        st["mc"] = st["mc_semi"] if SEMI_MDP else st["mc_standard"]

    mc_mean_values = np.array([s["mc"] for s in mc_states])
    print(f"\n{len(mc_states)} total states ({num_cached} cached, {len(mc_states) - num_cached} generated)")
    print(f"Using {'semi-MDP' if SEMI_MDP else 'standard MDP'} ground truth")
    print(f"Cache: {mc_cache_path}")

Loaded state_0.npz
Loaded state_1.npz
Loaded state_2.npz
Loaded state_3.npz
Loaded state_4.npz
Loaded state_5.npz
Loaded state_6.npz
Loaded state_7.npz
Loaded state_8.npz
Loaded state_9.npz
Loaded state_10.npz
Loaded state_11.npz
Loaded state_12.npz
Loaded state_13.npz
Loaded state_14.npz
Loaded state_15.npz
Loaded state_16.npz
Loaded state_17.npz
Loaded state_18.npz
Loaded state_19.npz
Loaded state_20.npz
Loaded state_21.npz
Loaded state_22.npz
Loaded state_23.npz
Loaded state_24.npz
Loaded state_25.npz
Loaded state_26.npz
Loaded state_27.npz
Loaded state_28.npz
Loaded state_29.npz
Loaded state_100.npz
Loaded state_101.npz
Loaded state_102.npz
Loaded state_103.npz
Loaded state_104.npz
Loaded state_105.npz
Loaded state_106.npz
Loaded state_107.npz
Loaded state_108.npz
Loaded state_109.npz
Found 40 cached states in mc_cache_inline/rp-1.0_g0.99_N4_d1000_t5_r200_seed42069

40 total states (40 cached, 0 generated)
Using semi-MDP ground truth
Cache: mc_cache_inline/rp-1.0_g0.99_N4_d1000_t5_

In [4]:
# =============================================================================
# PLOT MC DISTRIBUTION (first state)
# =============================================================================

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
if not PHASE2_EVAL:
    st0 = mc_states[0]
    fig, axes = plt.subplots(2, NUM_AGENTS, figsize=(4 * NUM_AGENTS, 8), sharex=False)

    for scheme, row, dist_key, mc_key in [
        ("Standard MDP", 0, "dist_standard", "mc_standard"),
        ("Semi-MDP", 1, "dist_semi", "mc_semi"),
    ]:
        for i in range(NUM_AGENTS):
            ax = axes[row, i]
            data = st0[dist_key][i]  # (num_traj,)
            mu, sigma = data.mean(), data.std()

            ax.hist(data, bins=20, density=True, alpha=0.6, color=f"C{i}")

            # Fit normal
            x = np.linspace(data.min(), data.max(), 100)
            ax.plot(x, stats.norm.pdf(x, mu, sigma), 'k-', linewidth=1.5)

            ax.axvline(st0[mc_key][i], color='red', linestyle='--', linewidth=1.5, label=f"mean={st0[mc_key][i]:.4f}")
            ax.set_title(f"Agent {i} — {scheme}\nμ={mu:.4f}, σ={sigma:.4f}, n={len(data)}")
            ax.legend(fontsize=8)
            if i == 0:
                ax.set_ylabel("Density")

    fig.suptitle(f"MC Return Distribution (State 0, {MC_TRAJ} traj × {MC_ROLLOUTS} rollouts)", fontsize=13)
    fig.tight_layout()

    plot_path = os.path.join(OUTPUT_PATH, "mc_distribution_state0.png")
    os.makedirs(OUTPUT_PATH, exist_ok=True)
    fig.savefig(plot_path, dpi=200, bbox_inches="tight")
    print(f"Saved to {plot_path}")
    plt.show()

Saved to outputs/mc_distribution_state0.png


In [5]:
# =============================================================================
# CREATE MODELS (one per agent)
# =============================================================================

set_all_seeds(SEED)

models = []
optimizers = []
for i in range(NUM_AGENTS):
    m = ValueNetwork(INPUT_DIM, MLP_LAYERS).to(device)
    models.append(m)
    optimizers.append(torch.optim.SGD(m.parameters(), lr=LR))

total_params = sum(p.numel() for p in models[0].parameters())
print(f"Model: {MLP_LAYERS}, {total_params} params per agent")
print(models[0])

# LR scheduler (linear decay over 80%, then hold)
decay_steps = int(TRAIN_SECONDS * LR_DECAY_FRAC)
min_factor = (LR_MIN / LR) if LR > 0 else 0.0
schedulers = [
    torch.optim.lr_scheduler.LambdaLR(
        opt, lr_lambda=lambda s: lr_factor(s, decay_steps, min_factor)
    )
    for opt in optimizers
]


Model: [16, 16, 16, 16], 1729 params per agent
ValueNetwork(
  (net): Sequential(
    (0): Linear(in_features=55, out_features=16, bias=True)
    (1): LeakyReLU(negative_slope=0.01)
    (2): Linear(in_features=16, out_features=16, bias=True)
    (3): LeakyReLU(negative_slope=0.01)
    (4): Linear(in_features=16, out_features=16, bias=True)
    (5): LeakyReLU(negative_slope=0.01)
    (6): Linear(in_features=16, out_features=16, bias=True)
    (7): LeakyReLU(negative_slope=0.01)
    (8): Linear(in_features=16, out_features=1, bias=True)
  )
)


In [6]:

# =============================================================================
# TRAINING LOOP
# =============================================================================

os.makedirs(OUTPUT_PATH, exist_ok=True)
csv_path = os.path.join(OUTPUT_PATH, f"phase1_{TAG}.csv")
meta_path = csv_path.replace('.csv', '_metadata.json')
ckpt_dir = os.path.join(OUTPUT_PATH, f"checkpoints_{TAG}")
os.makedirs(ckpt_dir, exist_ok=True)

if PHASE2_EVAL:
    metadata = {
    "phase": 1, "R_PICKER": R_PICKER, "R_OTHER": R_OTHER,
    "LAMBDA": LAMBDA, "MLP_LAYERS": MLP_LAYERS,
    "LR": LR, "LR_MIN": LR_MIN, "LR_DECAY_FRAC": LR_DECAY_FRAC,
    "TRAIN_SECONDS": TRAIN_SECONDS, "EVAL_FREQ": EVAL_FREQ,
    "SEED": SEED, "H": H, "W": W, "NUM_AGENTS": NUM_AGENTS,
    "GAMMA": GAMMA, "GAMMA_STEP": GAMMA_STEP,
    "K_NEAREST": K_NEAREST, "INPUT_DIM": INPUT_DIM, "L": L,
    "P_SPAWN": P_SPAWN, "P_DESPAWN": P_DESPAWN,
    }
else:
    # Metadata
    metadata = {
        "phase": 1, "R_PICKER": R_PICKER, "R_OTHER": R_OTHER,
        "LAMBDA": LAMBDA, "MLP_LAYERS": MLP_LAYERS,
        "LR": LR, "LR_MIN": LR_MIN, "LR_DECAY_FRAC": LR_DECAY_FRAC,
        "TRAIN_SECONDS": TRAIN_SECONDS, "EVAL_FREQ": EVAL_FREQ,
        "SEED": SEED, "H": H, "W": W, "NUM_AGENTS": NUM_AGENTS,
        "GAMMA": GAMMA, "GAMMA_STEP": GAMMA_STEP,
        "K_NEAREST": K_NEAREST, "INPUT_DIM": INPUT_DIM, "L": L,
        "P_SPAWN": P_SPAWN, "P_DESPAWN": P_DESPAWN,
        "num_mc_states": len(mc_states),
        "avg_magnitude_true_value": {
            f"agent_{i}": float(np.mean(np.abs(mc_mean_values[:, i])))
            for i in range(NUM_AGENTS)
        },
    }
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=4)

# Initialize environment
set_all_seeds(SEED)
reward_mod = Reward(R_PICKER, NUM_AGENTS)
env = Orchard(W, W, NUM_AGENTS, reward_mod, P_SPAWN, P_DESPAWN)
env.set_positions()

curr_state = None
actor_idx = None
first_write = True
prev_weights = {i: [] for i in range(NUM_AGENTS)}
total_grad_steps = 0
t0 = time.time()
gamma_m0 = GAMMA_SEMI if SEMI_MDP else GAMMA_STEP   # mode 0→1
gamma_m1 = 1.0 if SEMI_MDP else GAMMA_STEP           # mode 1→0
def get_epsilon(second):
    if not PHASE2_EVAL:
        return 0.0
    return EPSILON_END + max(0, 1 - second / (TRAIN_SECONDS * EPSILON_DECAY_FRAC)) * (EPSILON_START - EPSILON_END)

# Per-agent batch buffers
if TD_MODE == "td0":
    batch_states = {i: [] for i in range(NUM_AGENTS)}
    batch_new_states = {i: [] for i in range(NUM_AGENTS)}
    batch_rewards = {i: [] for i in range(NUM_AGENTS)}
    batch_gammas = {i: [] for i in range(NUM_AGENTS)}
elif TD_MODE == "tdlambda":
    BATCH = 2 * NUM_AGENTS                    # transitions per second
    buf_size = BATCH * WINDOW_SEC              # buffer capacity (transitions)
    buf_enc = {i: [] for i in range(NUM_AGENTS)}  # encodings: length n+1 when full
    buf_r = {i: [] for i in range(NUM_AGENTS)}     # rewards: length n when full
    buf_gamma = []                                  # discounts: length n, shared across agents
else:
    raise ValueError(f"Invalid TD_MODE: {TD_MODE}")

diag = {i: {"grad_norm": [], "target_max": [], "loss": [],
            "v_max": [], "barr_max": [], "delta_max": []}
        for i in range(NUM_AGENTS)}

def flush_diag():
    """Return max of each diagnostic since last flush, then reset."""
    out = {}
    for i in range(NUM_AGENTS):
        for key in diag[i]:
            vals = diag[i][key]
            if vals:
                out[f"a{i}_{key}"] = float(np.nanmax(vals))
            else:
                out[f"a{i}_{key}"] = float("nan")
            diag[i][key].clear()
    return out

print(f"Training: {TRAIN_SECONDS} seconds, λ={LAMBDA}")
print(f"Logging to: {csv_path}")

for second in range(TRAIN_SECONDS):
    if TD_MODE == "td0":
        # --- Collect experiences for this section ---
        for step in range(-1, NUM_AGENTS):
            act_override = None
            if PHASE2_EVAL and step != -1:
                if random.random() < get_epsilon(second):
                    act_override = random_action(curr_state["agent_positions"][actor_idx], W)
                else:
                    act_override = greedy_action(curr_state, actor_idx, models, device)
                    
            final_state, semi_state, res, actor_idx = transition(
                step, curr_state, env, actor_idx, action_override=act_override
            )
            if step != -1:
                for i in range(NUM_AGENTS):
                    enc_old = encode_state(curr_state, i)
                    enc_semi = encode_state(semi_state, i)
                    enc_final = encode_state(final_state, i)

                    batch_states[i].append(enc_old)
                    batch_new_states[i].append(enc_semi)
                    batch_rewards[i].append(0.0)
                    batch_gammas[i].append(gamma_m0)

                    batch_states[i].append(enc_semi)
                    batch_new_states[i].append(enc_final)
                    batch_rewards[i].append(res.reward_vector[i])
                    batch_gammas[i].append(gamma_m1)

            curr_state = final_state

        # --- Batch TD(0) update ---
        for i in range(NUM_AGENTS):
            B = len(batch_states[i])
            states_t = torch.as_tensor(
                np.stack(batch_states[i]), device=device
            ).view(B, -1)
            nxt_t = torch.as_tensor(
                np.stack(batch_new_states[i]), device=device
            ).view(B, -1)
            rewards_t = torch.as_tensor(
                np.array(batch_rewards[i], dtype=np.float64), device=device
            )

            pred = models[i](states_t).squeeze(-1)
            with torch.no_grad():
                v_next = models[i](nxt_t).squeeze(-1)
                gammas_t = torch.as_tensor(
                np.array(batch_gammas[i], dtype=np.float64), device=device
                )           
                target = rewards_t + gammas_t * v_next

            loss = torch.nn.functional.mse_loss(pred, target)
            optimizers[i].zero_grad()
            loss.backward()
            if ADD_GRAD_CLIP:
                torch.nn.utils.clip_grad_norm_(models[i].parameters(), CLIP_GRAD_NORM)
            optimizers[i].step()
            schedulers[i].step()

            batch_states[i].clear()
            batch_new_states[i].clear()
            batch_rewards[i].clear()
            batch_gammas[i].clear()

        total_grad_steps += 1

    elif TD_MODE == "tdlambda":
        # --- Collect 2N sequential transitions ---
        for step in range(-1, NUM_AGENTS):
            act_override = None
            if PHASE2_EVAL and step != -1:
                if random.random() < get_epsilon(second):
                    act_override = random_action(curr_state["agent_positions"][actor_idx], W)
                else:
                    act_override = greedy_action(curr_state, actor_idx, models, device)

            final_state, semi_state, res, actor_idx = transition(
                step, curr_state, env, actor_idx, action_override=act_override
            )
            if step == -1:
                curr_state = final_state
                # Seed buffer with initial state on first second only
                if second == 0:
                    for i in range(NUM_AGENTS):
                        buf_enc[i].append(encode_state(curr_state, i))
                continue
            else:
                for i in range(NUM_AGENTS):
                    # Mode 0 → Mode 1
                    buf_enc[i].append(encode_state(semi_state, i))
                    buf_r[i].append(0.0)
                    # Mode 1 → Mode 0
                    buf_enc[i].append(encode_state(final_state, i))
                    buf_r[i].append(res.reward_vector[i])
                buf_gamma.append(gamma_m0)
                buf_gamma.append(gamma_m1)

            curr_state = final_state

        # --- TD(λ) update when buffer is full ---
        while len(buf_gamma) >= buf_size:
            n = len(buf_gamma)

            for i in range(NUM_AGENTS):
                enc_np = np.stack(buf_enc[i])                          # (n+1, INPUT_DIM)
                r_np = np.array(buf_r[i], dtype=np.float64)            # (n,)
                g_np = np.array(buf_gamma, dtype=np.float64)           # (n,)

                # No-grad forward pass on all n+1 states
                with torch.no_grad():
                    all_v = models[i](
                        torch.as_tensor(enc_np, device=device)
                    ).cpu().numpy()                                    # (n+1,)

                # TD errors: δ[j] = r[j] + γ[j] * V(s[j+1]) - V(s[j])
                delta = r_np + g_np * all_v[1:] - all_v[:-1]          # (n,)

                # Backward scan: B[j] = δ[j] + λ * γ[j] * B[j+1]
                B_arr = np.zeros(n, dtype=np.float64)
                B_arr[n - 1] = delta[n - 1]
                for j in range(n - 2, -1, -1):
                    B_arr[j] = delta[j] + LAMBDA * g_np[j] * B_arr[j + 1]

                # Targets for oldest BATCH states (no-grad)
                targets = all_v[:BATCH] + B_arr[:BATCH]
                
                if CLAMP_TARGETS:
                    targets = np.clip(targets, -100, 100)
                
                # --- Diagnostics ---
                target_abs_max = np.max(np.abs(targets))
                barr_abs_max = np.max(np.abs(B_arr[:BATCH]))
                v_abs_max = np.max(np.abs(all_v))
                delta_abs_max = np.max(np.abs(delta))
                targets_finite = bool(np.all(np.isfinite(targets)))

                # With-grad forward pass on oldest BATCH states
                pred = models[i](torch.as_tensor(enc_np[:BATCH], device=device))
                target_t = torch.as_tensor(targets, device=device)

                loss = torch.nn.functional.mse_loss(pred, target_t)
                optimizers[i].zero_grad()
                loss.backward()
                
                grad_norm = torch.nn.utils.clip_grad_norm_(models[i].parameters(), float('inf'))
                diag[i]["grad_norm"].append(float(grad_norm))
                diag[i]["target_max"].append(float(target_abs_max))
                diag[i]["loss"].append(float(loss.item()) if torch.isfinite(loss) else float("nan"))
                diag[i]["v_max"].append(float(v_abs_max))
                diag[i]["barr_max"].append(float(barr_abs_max))
                diag[i]["delta_max"].append(float(delta_abs_max))
                
                if not targets_finite or not torch.isfinite(loss) or target_abs_max > 50 or grad_norm > 1000:
                    print(f"  WARN second={second} agent={i} | "
                          f"target_max={target_abs_max:.1f} barr_max={barr_abs_max:.1f} "
                          f"v_max={v_abs_max:.1f} delta_max={delta_abs_max:.1f} "
                          f"loss={loss.item():.4f} grad_norm={grad_norm:.1f} "
                          f"targets_finite={targets_finite}")

                if ADD_GRAD_CLIP:
                    torch.nn.utils.clip_grad_norm_(models[i].parameters(), CLIP_GRAD_NORM)
                optimizers[i].step()
                schedulers[i].step()

            # Pop oldest BATCH entries (keep state at position BATCH as new start)
            for i in range(NUM_AGENTS):
                buf_enc[i] = buf_enc[i][BATCH:]
                buf_r[i] = buf_r[i][BATCH:]
            buf_gamma = buf_gamma[BATCH:]

            total_grad_steps += 1

    # --- Evaluate ---
    if (second + 1) % EVAL_FREQ == 0:
        if PHASE2_EVAL:
            metrics = evaluate_policies(
                models, POLICY_EVAL_EPISODES, POLICY_EVAL_SECS,
                reward_mod, SEED + second, device,
            )
        else:
            metrics = evaluate_on_mc(models, mc_states, device)
            
        current_lr = optimizers[0].param_groups[0]['lr']
        row = {'second': second + 1, 'total_grad_steps': total_grad_steps,
               'current_lr': current_lr, 'memory_mb': get_memory_mb()}
        row.update(metrics)
        row.update(flush_diag())
        for i in range(NUM_AGENTS):
            ws, prev_weights[i] = get_weight_stats(models[i], prev_weights[i])
            for k, v in ws.items():
                row[f'a{i}_{k}'] = v

        pd.DataFrame([row]).to_csv(
            csv_path, mode='w' if first_write else 'a',
            header=first_write, index=False,
        )
        first_write = False

    if (second + 1) % PRINT_FREQ == 0:
        elapsed = time.time() - t0
        current_lr = optimizers[0].param_groups[0]['lr']
        if PHASE2_EVAL:
            metrics = evaluate_policies(
                models, POLICY_EVAL_EPISODES, POLICY_EVAL_SECS,
                reward_mod, SEED + second, device,
            )
            print(f"second {second+1:>8d} | grad_steps={total_grad_steps} | "
                  f"greedy={metrics['greedy_picks_per_sec']:.2f} | "
                  f"nearest={metrics['nearest_picks_per_sec']:.2f} | "
                  f"random={metrics['random_picks_per_sec']:.2f} | "
                  f"eps={get_epsilon(second):.3f} | "
                  f"lr={current_lr:.2e} | {elapsed:.0f}s")
        else:
            metrics = evaluate_on_mc(models, mc_states, device)
            print(f"second {second+1:>8d} | grad_steps={total_grad_steps} | "
                  f"pct_err={metrics['pct_err_total']:.2f}% | "
                  f"mae={metrics['mae_total']:.4f} | "
                  f"bias={metrics['bias_total']:.4f} | "
                  f"lr={current_lr:.2e} | {elapsed:.0f}s")

    # --- Checkpoint ---
    if (second + 1) % CHECKPOINT_FREQ == 0:
        for i in range(NUM_AGENTS):
            torch.save(models[i].state_dict(),
                       os.path.join(ckpt_dir, f"agent{i}_tick{second+1}.pt"))

elapsed = time.time() - t0
print(f"\nDone in {elapsed:.1f}s ({total_grad_steps} grad steps)")


Training: 1000000 seconds, λ=0.8
Logging to: outputs/phase1_td_lam0.8_lr0.01_rp-1.0_semiTruegamma0.99.csv
  WARN second=219 agent=2 | target_max=265.5 barr_max=20.7 v_max=277.4 delta_max=18.6 loss=135.9645 grad_norm=3877.5 targets_finite=True
  WARN second=220 agent=2 | target_max=127.3 barr_max=14.3 v_max=136.1 delta_max=23.5 loss=66.3199 grad_norm=2093.8 targets_finite=True
  WARN second=231 agent=2 | target_max=245.3 barr_max=23.7 v_max=259.0 delta_max=26.0 loss=134.7331 grad_norm=1263.7 targets_finite=True
  WARN second=232 agent=2 | target_max=27490.7 barr_max=1910.0 v_max=28727.6 delta_max=2721.3 loss=979704.8860 grad_norm=31595354.6 targets_finite=True
  WARN second=233 agent=2 | target_max=6717973860183485062316032.0 barr_max=566439297552463312191488.0 v_max=7102694564957468852486144.0 delta_max=683405092801250888515584.0 loss=65977193400888869464852954241445871636929576960.0000 grad_norm=70000912033528619013642518274607971896918016.0 targets_finite=True
  WARN second=234 agent

/tmp/ipykernel_260301/4082414208.py:87: RuntimeWarning: All-NaN axis encountered
  out[f"a{i}_{key}"] = float(np.nanmax(vals))


  WARN second=250 agent=2 | target_max=nan barr_max=nan v_max=nan delta_max=nan loss=nan grad_norm=nan targets_finite=False
  WARN second=251 agent=2 | target_max=nan barr_max=nan v_max=nan delta_max=nan loss=nan grad_norm=nan targets_finite=False
  WARN second=252 agent=2 | target_max=nan barr_max=nan v_max=nan delta_max=nan loss=nan grad_norm=nan targets_finite=False
  WARN second=253 agent=2 | target_max=nan barr_max=nan v_max=nan delta_max=nan loss=nan grad_norm=nan targets_finite=False
  WARN second=254 agent=2 | target_max=nan barr_max=nan v_max=nan delta_max=nan loss=nan grad_norm=nan targets_finite=False
  WARN second=255 agent=2 | target_max=nan barr_max=nan v_max=nan delta_max=nan loss=nan grad_norm=nan targets_finite=False
  WARN second=256 agent=2 | target_max=nan barr_max=nan v_max=nan delta_max=nan loss=nan grad_norm=nan targets_finite=False
  WARN second=257 agent=2 | target_max=nan barr_max=nan v_max=nan delta_max=nan loss=nan grad_norm=nan targets_finite=False
  WARN s

KeyboardInterrupt: 

In [ ]:
print("=" * 60)
print("FINAL RESULTS")
print("=" * 60)

if PHASE2_EVAL:
    metrics = evaluate_policies(
        models, POLICY_EVAL_EPISODES, POLICY_EVAL_SECS,
        reward_mod, SEED, device,
    )
    print(f"Greedy picks/sec:  {metrics['greedy_picks_per_sec']:.4f}")
    print(f"Nearest picks/sec: {metrics['nearest_picks_per_sec']:.4f}")
    print(f"Random picks/sec:  {metrics['random_picks_per_sec']:.4f}")
else:
    metrics = evaluate_on_mc(models, mc_states, device)

    for i in range(NUM_AGENTS):
        print(f"\nAgent {i}:")
        print(f"  MAE:       {metrics[f'a{i}_mae']:.6f}")
        print(f"  Bias:      {metrics[f'a{i}_bias']:.6f}")
        print(f"  % Error:   {metrics[f'a{i}_pct_error']:.2f}%")
        print(f"  Mean Pred: {metrics[f'a{i}_mean_pred']:.6f}")
        print(f"  Mean True: {metrics[f'a{i}_mean_true']:.6f}")

    print(f"\nOverall MAE:     {metrics['mae_total']:.6f}")
    print(f"Overall % Error: {metrics['pct_err_total']:.2f}%")

    if metrics['pct_err_total'] < 10.0:
        print("*** PASS: <10% error ***")
    else:
        print("*** FAIL: >=10% error ***")